In [ ]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import yaml, pickle, glob, csv

In [ ]:
import sys
sys.path.append('./code')

In [ ]:
from models import XASGNN, SpectrumHead, XASLightningModule, MLPLightningModule
from data import XASGraphDataset, FeatureDataset, collate_fn

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import lightning.pytorch as pl
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar     

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
GRID=np.linspace(9500-600, 9650-600, 1000)
energy_grid = GRID[333:733:2]
ROOT_PATH = './'

In [ ]:
with open(ROOT_PATH + "/configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

## 1. Load data

In [ ]:
train_structures = pickle.load(open(ROOT_PATH + '/dataset/train_structures.pkl', 'rb'))
train_spectra = torch.load(ROOT_PATH + "/dataset/train_spectra.pt")
val_structures = pickle.load(open(ROOT_PATH + '/dataset/val_structures.pkl', 'rb'))
val_spectra = torch.load(ROOT_PATH + "/dataset/val_spectra.pt")
# test_structures = pickle.load(open(ROOT_PATH + '/dataset/test_structures.pkl', 'rb'))
# test_spectra = torch.load(ROOT_PATH + "/dataset/test_spectra.pt")

In [ ]:
cutoff = config['gnn']['cutoff']

In [ ]:
train_dataset = XASGraphDataset(train_structures, train_spectra, cutoff=cutoff)
val_dataset = XASGraphDataset(val_structures, val_spectra, cutoff=cutoff)
# test_dataset = XASGraphDataset(test_structures, test_spectra, cutoff=cutoff)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=config['training']['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=config['training']['batch_size'], collate_fn=collate_fn)
# test_loader = DataLoader(test_dataset, batch_size=config['training']['batch_size'], collate_fn=collate_fn)

## 2. Train model

### 2.1. Train GNN model

#### 2.1.1. Train

In [ ]:
csv_logger = CSVLogger("lightning_logs", name="gnn-%s-%s"%(config['gnn']['cutoff'],config['gnn']['nblocks']))
early_stop_callback = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    min_delta=1e-5,       
    patience=20,          
    mode='min'             
)

In [ ]:
gnn_config = dict(
    nblocks = config['gnn']['nblocks'], 
    cutoff = config['gnn']['cutoff'], 
    threebody_cutoff = config['gnn']['threebody_cutoff']
)

head_config = dict(
    hidden_dims = [64, 64], 
    output_size = config['head']['output_size'], 
    drop_rate = config['head']['drop_rate'], 
)

In [ ]:
model = XASLightningModule(gnn_config, head_config, learning_rate=config['training']['lr'])

In [ ]:
# =============================
#### Modify max_epochs here ### 
# =============================
trainer = pl.Trainer(max_epochs=10, accelerator="gpu", devices=1, 
                     callbacks=[early_stop_callback, 
                                TQDMProgressBar(refresh_rate=0), 
                                pl.callbacks.ModelCheckpoint(monitor='val_loss', save_top_k=1)
                               ], 
                     log_every_n_steps=0,          # disables step-level logging
                     logger=csv_logger,                     
                    )
trainer.fit(model, train_loader, val_loader)

#### 2.1.2. Predict 

In [ ]:
loadpath = './lightning_logs/gnn-4.0-3/version_0/'
model = XASLightningModule.load_from_checkpoint(glob.glob(loadpath+'checkpoints/*.ckpt')[0])

In [ ]:
model = model.to(device)
model.eval()

predictions = []
val_loss = 0.0

with torch.no_grad():   # disables gradient tracking
    for g, _, spectra in val_loader:
        g, spectra = g.to(device), spectra.to(device)
        y_hat = model(g)

        predictions.append(y_hat.cpu())

        loss = torch. nn.MSELoss()(y_hat, spectra)
        val_loss += loss.item() * spectra.size(0)  # sum over batch

# concatenate all predictions into one tensor
predictions = torch.cat(predictions, dim=0)

print(f"Validation loss: {val_loss / len(val_loader.dataset)}")

#### 2.1.3 Plot learning curve

In [ ]:
metrics = []
with open(loadpath + '/metrics.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        metrics.append(row)

In [ ]:
epochs = []
val_losses = []
train_losses = []
for row in metrics:
    if row["val_loss"]:  # not empty
        epochs.append(int(row["epoch"]))
        val_losses.append(float(row["val_loss"]))
    if row["train_loss"]:  # not empty
        train_losses.append(float(row["train_loss"]))    

In [ ]:
plt.plot(epochs, train_losses, lw=1, label='train, $\mathcal{L}$')
plt.plot(epochs, val_losses, lw=1, linestyle='--', label='val, $\mathcal{L}$')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.tick_params(bottom=True, top=True, left=True, right=True)
plt.legend(fontsize=15, frameon=False, loc=(0.5, 0.55))
# plt.yscale('log')  # Set the y-axis to a logarithmic scale
# plt.savefig('./figures/loss_curve.png', bbox_inches='tight', dpi=250)

### 2.2. Train MLP model

#### 2.2.1 Load pre-trained gnn and featurize

In [ ]:
def cache_features(gnn, dataloader, save_path="absorber_features.pt", device="cuda"):
    gnn.eval().to(device)
    feats_all, spectra_all = [], []

    with torch.no_grad():
        for g, _, spectra in dataloader:
            g = g.to(device)
            spectra = spectra.to(device)
            feats = gnn(g)  # (B, d)
            feats_all.append(feats.cpu())
            spectra_all.append(spectra.cpu())

    feats_all = torch.cat(feats_all, dim=0)
    spectra_all = torch.cat(spectra_all, dim=0)
    torch.save((feats_all, spectra_all), save_path)
    print(f"Saved cached features to {save_path}")

In [ ]:
loadpath = './lightning_logs/gnn-4.0-3/version_0/'
model = XASLightningModule.load_from_checkpoint(glob.glob(loadpath+'checkpoints/*.ckpt')[0])

In [ ]:
# Save to cached features
features_file = "absorber_features.pt"
cache_features(model.gnn, train_loader, save_path=loadpath+"/train_" + features_file, device=device)
cache_features(model.gnn, val_loader, save_path=loadpath+"/val_" + features_file, device=device)

In [ ]:
# Load cached features
features_file = "absorber_features.pt"

train_feats, train_spectra = torch.load(loadpath+"/train_" + features_file)
val_feats, val_spectra = torch.load(loadpath+"/val_" + features_file)

train_dataset_feat = FeatureDataset(train_feats, train_spectra)
val_dataset_feat = FeatureDataset(val_feats, val_spectra)

train_loader_feat = DataLoader(train_dataset_feat, batch_size=config['training']['batch_size'], shuffle=True)
val_loader_feat = DataLoader(val_dataset_feat, batch_size=config['training']['batch_size'])

#### 2.2.2 Train MLP

In [ ]:
csv_logger = CSVLogger("lightning_logs", name="mlp-%s-%s"%(config['gnn']['cutoff'],config['gnn']['nblocks']))
early_stop_callback = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    min_delta=1e-6,         
    patience=100,             
    mode='min'             
)

In [ ]:
head_config = dict(
    hidden_dims=config['head']['hidden_dims'], 
    output_size=config['head']['output_size'], 
    drop_rate=config['head']['drop_rate'], 
)

mlp_model = MLPLightningModule(head_config, learning_rate=1e-3)

In [ ]:
# =============================
#### Modify max_epochs here ### 
# =============================
trainer = pl.Trainer(
    max_epochs=200,
    accelerator="gpu",
    callbacks=[early_stop_callback, 
            TQDMProgressBar(refresh_rate=0), 
            pl.callbacks.ModelCheckpoint(monitor='val_loss', save_top_k=1)
           ], 
    log_every_n_steps=0,
    logger=csv_logger, 
)

trainer.fit(mlp_model, train_loader_feat, val_loader_feat)

#### 2.2.3 Predict spectra from features

In [ ]:
loadpath = './lightning_logs/mlp-4.0-3/version_0/'
mlp_model = MLPLightningModule.load_from_checkpoint(glob.glob(loadpath+'checkpoints/*.ckpt')[0])

In [ ]:
mlp_model = mlp_model.to(device)
mlp_model.eval()

predictions = []
val_loss = 0.0

with torch.no_grad():   # disables gradient tracking
    for feats, spectra in val_loader_feat:
        feats, spectra = feats.to(device), spectra.to(device)
        y_hat = mlp_model(feats)

        predictions.append(y_hat.cpu())

        loss = torch.nn.MSELoss()(y_hat, spectra)
        val_loss += loss.item() * spectra.size(0)  # sum over batch

# concatenate all predictions into one tensor
predictions = torch.cat(predictions, dim=0)

print(f"Validation loss: {val_loss / len(val_loader_feat.dataset)}")

#### 2.2.4. Plot learning curve

In [ ]:
metrics = []
with open(loadpath+ '/metrics.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        metrics.append(row)

In [ ]:
epochs = []
val_losses = []
train_losses = []
for row in metrics:
    if row["val_loss"]:  # not empty
        epochs.append(int(row["epoch"]))
        val_losses.append(float(row["val_loss"]))
    if row["train_loss"]:  # not empty
        train_losses.append(float(row["train_loss"]))    

In [ ]:
plt.plot(epochs, train_losses, lw=1, label='train, $\mathcal{L}$')
plt.plot(epochs, val_losses, lw=1, linestyle='--', label='val, $\mathcal{L}$')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.tick_params(bottom=True, top=True, left=True, right=True)
plt.legend(fontsize=15, frameon=False, loc=(0.5, 0.55))
plt.yscale('log')  
# plt.savefig('./figures/loss_curve.png', bbox_inches='tight', dpi=250)

## 3. Load model and predict 

### 3.1 Load model

In [ ]:
loadpath = './lightning_logs/gnn-4.0-3/version_0/'
gnn_model = XASLightningModule.load_from_checkpoint(glob.glob(loadpath+'checkpoints/*.ckpt')[0])

In [ ]:
loadpath = './lightning_logs/mlp-4.0-3/version_0/'
mlp_model = MLPLightningModule.load_from_checkpoint(glob.glob(loadpath+'checkpoints/*.ckpt')[0])

In [ ]:
model = XASLightningModule(
    gnn_config=gnn_model.gnn_config,
    head_config=mlp_model.head_config,
    learning_rate=1e-5
)

model.gnn.load_state_dict(gnn_model.gnn.state_dict())
model.spectrum_head.load_state_dict(mlp_model.spectrum_head.state_dict())

### 3.2 predict

In [ ]:
model = model.to(device)
model.eval()

predictions = []
val_loss = 0.0

with torch.no_grad():   # disables gradient tracking
    for g, _, spectra in val_loader:
        g, spectra = g.to(device), spectra.to(device)
        y_hat = model(g)

        predictions.append(y_hat.cpu())

        loss = torch. nn.MSELoss()(y_hat, spectra)
        val_loss += loss.item() * spectra.size(0)  # sum over batch

# concatenate all predictions into one tensor
predictions = torch.cat(predictions, dim=0)

print(f"Validation loss: {val_loss / len(val_loader.dataset)}")

### 3.3. Plot decile curves

In [ ]:
performance = []
for ii, data in enumerate(val_dataset): 
    _, _, spectrum = data
    performance.append((nn.MSELoss()(spectrum, predictions[ii]), ii))

In [ ]:
performance.sort()

In [ ]:
# decimal plot for 10 different XAS 
idx = [i for loss, i in [performance[j] for j in range(20, 201, 20) ]]
for top_i, i in enumerate(idx):
    _, _, sp = val_dataset[i]
    pred = predictions[i]
    loss = nn.MSELoss()(pred/20, sp/20)
    plt.plot(energy_grid, pred.cpu().detach().numpy().flatten(), label='Predicted')
    plt.plot(energy_grid, sp.cpu().numpy().flatten(), label='True')
    plt.annotate(f'MSE = {loss:.3e}', xy=(8980, 0.1), fontsize=12)
    plt.legend()
    plt.show()


## Monitor GPU usage

In [ ]:
import gc

In [ ]:
torch.cuda.empty_cache()

In [ ]:
gc.collect()

In [ ]:
torch.cuda.ipc_collect()

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")